# Витрина транзакций ТЭ — 5. Первичный анализ по MCC

AI-14135. Распределение операций витрины `sandbox_ai.tmp_shestopalov_acq_mart` по MCC точки (`mcc_terminal`). Топ-10 кодов по обороту в профиле остаются отдельными, остальные склеиваются по `mcc_profile`. Код чека (`mcc_code`) в эту сводку не входит: его расхождение с точкой смотрит `mcc_mismatch`.

Что смотрим:
1. Сводку по каждому MCC: операции, оборот, клиенты, точки, терминалы, средний и медианный чек.
2. Концентрацию: сколько MCC дают 80%, 90% и 95% операций и оборота.
3. Малочисленные MCC: у скольких MCC мало клиентов. Для статистических тестов в AI-14135 нужно не меньше 30 клиентов в группе.
4. Подсказку для групп: что даёт укрупнение MCC по двум первым цифрам.

Операции со статусом `R` исключены, как в DAG и на дашборде. Чтобы их учесть, уберите условие `status <> 'R'` в запросе.

Названия MCC — словарь `MCC_NAME_RU` из `mcc_labels_enrich_drp.ipynb`, как на дашборде.

In [ ]:
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": "/home/jovyan/test_requests/tech.keytab",
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": "Shestopalov-VYur"},
)
imp._init_connection()

pd.set_option("display.float_format", lambda x: f"{x:,.0f}".replace(",", " ") if abs(x) >= 1000 else f"{x:.4f}")

In [ ]:
MCC_NAME_RU = {
  "149": "Прочие (код 149 — уточнить)",
  "1520": "Генеральные подрядчики - жилищное и торговое строительство",
  "1711": "Генеральные подрядчики по вентиляции, теплоснабжению, и водопроводу",
  "1731": "Подрядчики по электричеству",
  "1799": "Подрядчики, специализированная торговля - нигде более не классифицированные",
  "2741": "Разнообразные издательства/ печатное дело",
  "3543": "Отели Four Seasons",
  "4111": "Транспортировка - пригородные и локальные сезонные транспортные средства, включая электрички",
  "4112": "Пассажирские железнодорожные перевозки",
  "4131": "Автобусные линии, туристические автобусы",
  "4214": "Агентства по автотранспортным перевозкам, местные/дальные автогрузоперевоз-ки, компании попереезду и хранению, местная доставка",
  "4215": "Услуги курьера - по воздуху и на земле, агентство по отправке грузов",
  "4722": "Туроператоры, туристические агентства и организаторы экскурсий",
  "4812": "Магазины телекоммуникационного оборудования (телефоны, факсы, пейджеры и др.)",
  "4814": "Телекоммуникационные услуги, включая, но не ограничивая предоплаченые телефонные услуги и периодические телефонные услуги",
  "4816": "Компьютерная сеть/ информационные услуги",
  "4899": "Кабельные и другие платные телевизионные услуги",
  "4900": "Коммунальные услуги - электричество, газ, санитария, вода",
  "5021": "Офисная и торговая мебель",
  "5047": "Стоматологичееское/лабороторное/медицинское/офтальмологическое стацоинарное оборудование и устройства",
  "5051": "Центры и офисы работ по металлу",
  "5074": "Оборудование для водопровода и отопительной системы",
  "5169": "Химикалии и смежные вещества , не классифицированные ранее",
  "5192": "Книги, периодические издания и газеты",
  "5200": "Магазины бытовой техники",
  "5211": "Лесо- и строительный материал",
  "5231": "Розничная продажа стекла, красок и обоев",
  "5251": "Скобяные товары в розницу",
  "5261": "Садовые принадлежности (в том числе для ухода за газонами) в розницу",
  "5300": "Оптовики",
  "5309": "Беспошлинные магазины duty free",
  "5331": "Универсальные магазины",
  "5399": "Различные товары общего назначения",
  "5411": "Продуктовые магазины, супермаркеты",
  "5422": "Продажа свежего и мороженого мяса",
  "5441": "Кондитерские",
  "5451": "Продажа молочных продуктов в розницу",
  "5462": "Булочные, пекарни",
  "5499": "Различные продовольственные магазины - рынки, магазины со спец. ассортиментом, продажа полуфабрикатов, фирменных блюд, продажа с помощью торговых автоматов",
  "5511": "Легковой и грузовой транспорт - продажа, сервис, ремонт, запчасти и лизинг",
  "5533": "Запчасти, автомобильные аксессуары",
  "5541": "Бензиновые станции обслуживания (с или без дополнительного сервиса)",
  "5571": "Продажа мотоциклов",
  "5599": "Автодиллеры",
  "5611": "Мужская и подростковая одежда и аксессуары",
  "5621": "Магазины женской одежды",
  "5631": "Женские магазины аксессуаров",
  "5641": "Магазины детской одежды",
  "5651": "Семейные магазины одежды",
  "5655": "Спортивная одежда, одежда для верховой езды и езды на мотоцикле",
  "5661": "Магазины обуви",
  "5681": "Изготовление и продажа меховых изделий",
  "5691": "Магазины мужской и женской одежды",
  "5697": "Ателье",
  "5699": "Магазины одежды и аксессуаров",
  "5712": "Оборудование, мебель и бытовые принадлежности (кроме электрооборудования)",
  "5713": "Покрытия для пола",
  "5714": "Ткани, обивочный материал, гардины и портьеры, жалюзи",
  "5718": "Продажа каминов, экранов для каминов и аксессуаров",
  "5719": "Различные специализирован-ные магазины бытовых принадлежностей",
  "5722": "Другие магазины оборудования, домашние приборы",
  "5732": "Магазины электро-товаров",
  "5733": "Продажа музыкальных инструментов, фортепиано, нот",
  "5811": "Поставщики провизии",
  "5812": "Рестораны, места общественного питания",
  "5813": "Бары, коктейль-бары, дискотеки, ночные клубы и таверны - места продажи алкогольных напитков",
  "5814": "Рестораны быстрого обслуживания",
  "5912": "Аптеки, фармацевтические магазины",
  "5921": "Магазины с продажей спиртных напитков навынос (пиво, вино и ликер)",
  "5932": "Антикварные магазины - продажа, ремонт и услуги восстановления",
  "5940": "Веломагазины - продажа и обслуживание",
  "5941": "Магазины спортивных товаров",
  "5942": "Книжные магазины",
  "5943": "Магазины офисных, школьных принадлежностей, канцтоваров",
  "5944": "Магазины по продаже часов, ювелирных изделий и изделий из серебра",
  "5945": "Детские магазины",
  "5946": "Магазины по продаже камер, фильмов, видео и другого фотографического оборудования и приборов",
  "5947": "Магазины открыток, подарков, новинок и сувениров",
  "5948": "Магазины кожаных изделий, багажных сумок",
  "5949": "Магазины ткани, ниток рукоделия, шитья",
  "5950": "Магазины хрусталя и изделий из стекла",
  "5973": "Магазины религиозных товаров",
  "5976": "Ортопедические товары",
  "5977": "Магазины косметики",
  "5983": "Горючее топливо - уголь, нефть, разжиженный бензин, дрова",
  "5992": "Флористика",
  "5993": "Табачные магазины",
  "5994": "Дилеры по продаже печатной продукции",
  "5995": "Зоомагазины",
  "5999": "Различные магазины и специальные розничные магазины",
  "6211": "Ценные бумаги- брокеры/дилеры",
  "6300": "Продажа страхования, гарантированное размещение, премии",
  "6513": "Агенты и менеджеры по недвижимости — аренда",
  "7011": "Отели, сдаваемое жилье, не представленное в списке",
  "7032": "Спортивные и рекреационные лагеря",
  "7033": "Кемпинги и трейлерные парки",
  "7211": "Услуги прачечных",
  "7216": "Химчистки",
  "7221": "Фотостудии",
  "7230": "Парикмахерские, салоны красоты",
  "7251": "Ремонт обуви, чистка обуви и головных уборов",
  "7261": "Ритуальные услуги и крематории",
  "7297": "Массажная приемная. Терапевтические приемные, предлагающие услуги массажа.",
  "7298": "Салоны СПА-терапии",
  "7299": "Иной сервис",
  "7311": "Рекламные услуги",
  "7338": "Услуги копировальных центров",
  "7372": "Программирование, обработка данных, интегрированные системы, дизайн",
  "7379": "Ремонт и техническое обслуживание компьютерной техники",
  "7395": "Фотостудии, фотолаборатории",
  "7399": "Бизнес - сервис",
  "742": "Ветеринарные услуги",
  "7512": "Прокат автомобилей.",
  "7523": "Паркинги и гаражи",
  "7534": "Восстановление и ремонт покрышек",
  "7538": "Магазины автосервиса",
  "7542": "Автомойки",
  "7622": "Ремонт электроники",
  "7629": "Ремонт бытовой техники. ремонт электроприборы",
  "763": "Сельско-хозяйственные кооперативные общества",
  "7631": "Центры ремонта часов и чистки ювелирных изделий",
  "7699": "Общий ремонт",
  "7832": "Кинотеатры",
  "7922": "Агентства по продаже билетов",
  "7932": "Бильярд и Пул",
  "7941": "Спортивные поля, коммерческие спортивные состязания, профессиональные спортивные клубы, спортивные промоутеры",
  "7991": "Туристические аттракционы и шоу",
  "7996": "Луна-парки, карнавалы, цирки, предсказатели будущего",
  "7997": "Спортивные клубы и секции",
  "7999": "Спортивные развлечения",
  "8011": "Доктора - нигде ранее не классифицируемые",
  "8021": "Дантисты, ортодантисты",
  "8031": "Врачи узкой специализации (Остеопаты)",
  "8042": "Оптометристы, офтальмологи",
  "8043": "Оптика, оптические товары, и очки",
  "8050": "Уход и Средства Персонального обслуживания (санатории, роддома, дома пристарелых)",
  "8062": "Больницы",
  "8071": "Зубные и медицинские лаборатории",
  "8099": "Медицинские услуги, не представленные в списке",
  "8111": "Адвокаты, юридические услуги",
  "8211": "Школы, начальная и средняя",
  "8220": "Колледжи, университеты, профессиональные школы, и младшие колледжи",
  "8241": "Школы, корреспонденция",
  "8249": "Школы, торговые и профессионально-технические",
  "8299": "Услуги ухода за детьми",
  "8398": "Организации благотворительные и общественные службы",
  "8661": "Организации, религиозные",
  "8699": "Организации, членство - нигде ранее не классифицируемые",
  "8911": "Архитектурные, инжиниринговые услуги и услуги рассмотрения",
  "8931": "Бухгалтерский учет, ревизия, и бухгалтерские услуги"
}

## 1. Сводка по MCC

In [ ]:
with imp:
    mcc = imp.fetch("""
select
  cast(mcc_terminal as string) as mcc,
  count(*) as trx_cnt,
  sum(sum_t) as turnover,
  count(distinct inn) as inn_cnt,
  count(distinct retl_id) as retl_cnt,
  count(distinct id_pos_terminal) as terminal_cnt,
  avg(sum_t) as avg_amount,
  appx_median(sum_t) as median_amount
from sandbox_ai.tmp_shestopalov_acq_mart
where status <> 'R'
group by cast(mcc_terminal as string)
""")

num_cols = ["trx_cnt", "turnover", "inn_cnt", "retl_cnt", "terminal_cnt", "avg_amount", "median_amount"]
mcc[num_cols] = mcc[num_cols].astype(float)
mcc["mcc_name"] = mcc["mcc"].map(MCC_NAME_RU).fillna("нет в справочнике")
mcc = mcc.sort_values("trx_cnt", ascending=False).reset_index(drop=True)
mcc["trx_share"] = mcc["trx_cnt"] / mcc["trx_cnt"].sum()
mcc["turnover_share"] = mcc["turnover"] / mcc["turnover"].sum()
mcc["trx_share_cum"] = mcc["trx_share"].cumsum()

print("MCC всего:", len(mcc))
print("Операций:", mcc["trx_cnt"].sum(), " Оборот:", round(mcc["turnover"].sum()))
print("MCC без названия в справочнике:", (mcc["mcc_name"] == "нет в справочнике").sum())
display(mcc.head(30))

## 2. Концентрация

Сколько самых крупных MCC нужно, чтобы набрать долю операций или оборота.

In [ ]:
by_turnover = mcc.sort_values("turnover", ascending=False)
turnover_share_cum = by_turnover["turnover_share"].cumsum()

concentration = pd.DataFrame({
    "доля": ["80%", "90%", "95%"],
    "MCC по операциям": [(mcc["trx_share_cum"] < p).sum() + 1 for p in (0.8, 0.9, 0.95)],
    "MCC по обороту": [(turnover_share_cum < p).sum() + 1 for p in (0.8, 0.9, 0.95)],
})
display(concentration)

## 3. Малочисленные MCC

MCC по числу клиентов (ИНН). Группа с числом клиентов меньше 30 слишком мала для статистических тестов в AI-14135 — такие MCC надо объединять.

In [ ]:
mcc["inn_bucket"] = pd.cut(
    mcc["inn_cnt"],
    bins=[0, 1, 4, 9, 29, 99, float("inf")],
    labels=["1", "2-4", "5-9", "10-29", "30-99", "100+"],
)
buckets = mcc.groupby("inn_bucket", observed=False).agg(
    mcc_cnt=("mcc", "count"),
    trx_cnt=("trx_cnt", "sum"),
    turnover=("turnover", "sum"),
)
buckets["trx_share"] = buckets["trx_cnt"] / buckets["trx_cnt"].sum()
display(buckets)

print("MCC, где меньше 30 клиентов")
display(mcc[mcc["inn_cnt"] < 30][["mcc", "mcc_name", "inn_cnt", "retl_cnt", "trx_cnt", "turnover", "median_amount"]])

## 4. Подсказка для групп: две первые цифры MCC

Похожие виды деятельности стоят в MCC рядом: 54xx — продукты, 55xx — авто и АЗС, 56xx — одежда, 57xx — дом и электроника, 58xx — общепит, 59xx — прочая розница. Таблица показывает, сколько клиентов набирается в каждом таком укрупнении. Это не готовые группы, а основа для ручной нарезки.

In [ ]:
with imp:
    groups_2 = imp.fetch("""
select
  substring(lpad(cast(mcc_terminal as string), 4, '0'), 1, 2) as mcc_2,
  count(distinct mcc_terminal) as mcc_cnt,
  group_concat(distinct cast(mcc_terminal as string), ', ') as mcc_list,
  count(*) as trx_cnt,
  sum(sum_t) as turnover,
  count(distinct inn) as inn_cnt
from sandbox_ai.tmp_shestopalov_acq_mart
where status <> 'R'
group by substring(lpad(cast(mcc_terminal as string), 4, '0'), 1, 2)
order by trx_cnt desc
""")

groups_2[["trx_cnt", "turnover", "inn_cnt"]] = groups_2[["trx_cnt", "turnover", "inn_cnt"]].astype(float)
groups_2["trx_share"] = groups_2["trx_cnt"] / groups_2["trx_cnt"].sum()
display(groups_2)